# Ordered Set / SortedList Pattern

## Overview
SortedList (from `sortedcontainers`) is a powerful data structure that maintains elements in sorted order with efficient operations:
- **Insert/Remove**: O(log n)
- **Search/Bisect**: O(log n)
- **Index Access**: O(log n)

Much faster than repeatedly sorting a regular list (O(n log n) each time)!

## Key Operations
```python
from sortedcontainers import SortedList, SortedDict, SortedSet

sl = SortedList([3, 1, 2])  # Automatically sorted: [1, 2, 3]

# Add/Remove
sl.add(4)           # [1, 2, 3, 4]
sl.remove(2)        # [1, 3, 4]
sl.discard(10)      # No error if not exists

# Binary Search
sl.bisect_left(3)   # Index where 3 would be inserted (left side)
sl.bisect_right(3)  # Index where 3 would be inserted (right side)

# Access
sl[0]               # First element
sl[-1]              # Last element

# Pop
sl.pop(0)           # Remove and return first element
sl.pop(-1)          # Remove and return last element
```

## bisect_left vs bisect_right
```python
sl = SortedList([1, 3, 3, 3, 5])

sl.bisect_left(3)   # Returns 1 (insert before existing 3s)
sl.bisect_right(3)  # Returns 4 (insert after existing 3s)
sl.bisect_left(4)   # Returns 4 (same as bisect_right when not found)
```

## When to Use
- Maintaining sorted order with frequent insertions/deletions
- Finding kth smallest/largest dynamically
- Range queries and interval problems
- Calendar/scheduling problems
- Finding floor/ceiling values

## Resources
- [LeetCode Ordered Set Problem List](https://leetcode.com/problem-list/ordered-set/)
- [SortedContainers Documentation](https://grantjenks.com/docs/sortedcontainers/)
- [Using Python3 SortedDict Effectively](https://leetcode.com/discuss/study-guide/1515408/using-python3-sorteddict-effectively-floor-ceillings-minimum-maximum-etc)

---

In [ ]:
# Install sortedcontainers if needed
# !pip install sortedcontainers

---

# Problem 1: My Calendar I
**Difficulty**: Medium  
**Link**: https://leetcode.com/problems/my-calendar-i/

## Problem
Implement a `MyCalendar` class to store events. A new event can be added if it doesn't cause a double booking.

## Approach
Use SortedList to maintain intervals sorted by start time. Use `bisect_right` to find insertion point and check for overlaps.

## Pattern Recognition
- Need to maintain sorted intervals
- Check for overlaps efficiently
- Use bisect to find neighbors quickly

In [ ]:
# https://leetcode.com/problems/my-calendar-i/
from sortedcontainers import SortedList

class MyCalendar:
    def __init__(self):
        self.calendar = SortedList()  # Store (start, end) sorted by start

    def book(self, start: int, end: int) -> bool:
        """
        Time: O(log n) for bisect + O(log n) for add = O(log n)
        Space: O(n) for storing n events
        
        Example:
        calendar = [(10, 20), (30, 40)]
        book(15, 25) -> overlap with (10, 20) -> False
        book(25, 35) -> overlap with (30, 40) -> False
        book(5, 10)  -> no overlap -> True
        """
        # Find position where (start, end) would be inserted
        idx = self.calendar.bisect_right((start, end))
        
        # Check overlap with previous interval
        if idx > 0:
            prev_start, prev_end = self.calendar[idx - 1]
            if prev_end > start:  # Overlap: prev ends after new starts
                return False
        
        # Check overlap with next interval
        if idx < len(self.calendar):
            next_start, next_end = self.calendar[idx]
            if end > next_start:  # Overlap: new ends after next starts
                return False
        
        # No overlap, add the event
        self.calendar.add((start, end))
        return True

In [ ]:
from sortedcontainers import SortedDict


class Solution:
    def longestSubarray(self, nums: List[int], limit: int) -> int:
        # SortedDict to maintain the elements within the current window
        window = SortedDict()
        left = 0
        max_length = 0

        for right in range(len(nums)):
            if nums[right] in window:
                window[nums[right]] += 1

            else:
                window[nums[right]] = 1

            # Check if the absolute difference between the maximum and minimum values in the current window exceed the limit
            while window.peekitem(-1)[0] - window.peekitem(0)[0] > limit:
                # Remove the element at the left pointer from the SortedDict
                window[nums[left]] -= 1
                if window[nums[left]] == 0:
                    window.pop(nums[left])
                # Move the left pointer to the right to exclude the element causing the violation
                left += 1

            # Update maxLength with the length of the current valid window
            max_length = max(max_length, right - left + 1)

        return max_length

In [ ]:
# https://leetcode.com/problems/design-a-number-container-system/
class NumberContainers:

    def __init__(self):
        """
            find 10
                10 no t in number_index then  returns -1
            find 10
                10 in number_index then returns smallest of number_index

            change on existing index
            index 1 in index_number then get old_number and use index_number to get array of index to edit it
        """
        self.number_index = {} # 10: [1,2,3,5]
        self.index_number = {} # 2:10, 1:10, 3:10, 5:10 --> 1:20

    def change(self, index: int, number: int) -> None:
        if index in self.index_number:
            old_number = self.index_number[index]
            old_indexes = self.number_index[old_number]
            i = bisect.bisect_left(old_indexes, index)

            old_indexes.pop(i)
            if len(old_indexes) == 0:
                del self.number_index[old_number]
        
        self.index_number[index] = number
        if number not in self.number_index:
            self.number_index[number] = [index]
            return
        
        i = bisect.bisect_left(self.number_index[number], index)

        if i == len(self.number_index[number]) or self.number_index[number][i] != index:
            self.number_index[number].insert(i, index)
        
        

    def find(self, number: int) -> int:
        if number not in self.number_index:
            return -1
        return self.number_index[number][0]

---

# Problem 2: Find Median from Data Stream
**Difficulty**: Hard  
**Link**: https://leetcode.com/problems/find-median-from-data-stream/

## Problem
Design a data structure that supports:
- `addNum(int num)`: Add a number to the data structure
- `findMedian()`: Return the median of all elements

## Approach
SortedList maintains elements in sorted order, making median calculation trivial with O(1) access.

## Pattern Recognition
- Need dynamic sorted order
- Frequent insertions + median queries
- SortedList simpler than two-heap approach

In [ ]:
# https://leetcode.com/problems/find-median-from-data-stream/
from sortedcontainers import SortedList

class MedianFinder:
    def __init__(self):
        self.sorted_list = SortedList()

    def addNum(self, num: int) -> None:
        """
        Time: O(log n) - binary search insertion
        """
        self.sorted_list.add(num)

    def findMedian(self) -> float:
        """
        Time: O(1) - direct index access
        
        Example:
        [1, 2, 3, 4, 5] -> median = 3 (middle)
        [1, 2, 3, 4]    -> median = (2 + 3) / 2 = 2.5
        """
        n = len(self.sorted_list)
        
        if n % 2 == 1:
            # Odd: return middle element
            return float(self.sorted_list[n // 2])
        else:
            # Even: return average of two middle elements
            mid1 = self.sorted_list[n // 2 - 1]
            mid2 = self.sorted_list[n // 2]
            return (mid1 + mid2) / 2.0

---

# Problem 3: Contains Duplicate III
**Difficulty**: Hard  
**Link**: https://leetcode.com/problems/contains-duplicate-iii/

## Problem
Given an array, return true if there are two distinct indices i and j such that:
- `abs(nums[i] - nums[j]) <= valueDiff`
- `abs(i - j) <= indexDiff`

## Approach
Use sliding window + SortedList. Maintain a window of size `indexDiff` and use bisect to find values within `valueDiff`.

## Pattern Recognition
- Sliding window with sorted order
- Range queries within window
- Efficient add/remove from window

In [ ]:
# https://leetcode.com/problems/contains-duplicate-iii/
from sortedcontainers import SortedList
from typing import List

class Solution:
    def containsNearbyAlmostDuplicate(self, nums: List[int], indexDiff: int, valueDiff: int) -> bool:
        """
        Time: O(n log k) where k = indexDiff (window size)
        Space: O(k)
        
        Example:
        nums = [1, 5, 9, 1, 5, 9], indexDiff = 2, valueDiff = 3
        
        Window [1, 5]:      check if any value in [1-3, 1+3] = [-2, 4] -> 1 found!
        """
        if indexDiff <= 0 or valueDiff < 0:
            return False
        
        window = SortedList()
        
        for i, num in enumerate(nums):
            # Maintain window size <= indexDiff
            if i > indexDiff:
                window.remove(nums[i - indexDiff - 1])
            
            # Find the leftmost position >= (num - valueDiff)
            pos = window.bisect_left(num - valueDiff)
            
            # Check if element at pos is within valueDiff
            if pos < len(window) and abs(window[pos] - num) <= valueDiff:
                return True
            
            window.add(num)
        
        return False

---

# Problem 4: Range Module
**Difficulty**: Hard  
**Link**: https://leetcode.com/problems/range-module/

## Problem
Track ranges of numbers. Support:
- `addRange(left, right)`: Add interval [left, right)
- `queryRange(left, right)`: Check if interval is tracked
- `removeRange(left, right)`: Stop tracking interval

## Approach
Use SortedList to store non-overlapping intervals. Merge/split intervals as needed.

## Pattern Recognition
- Interval merging/splitting
- Need efficient range lookups
- Dynamic interval management

In [ ]:
# https://leetcode.com/problems/range-module/
from sortedcontainers import SortedList

class RangeModule:
    def __init__(self):
        self.ranges = SortedList()  # Store non-overlapping [left, right) intervals

    def addRange(self, left: int, right: int) -> None:
        """
        Time: O(n) worst case for merging
        
        Example:
        ranges = [(1, 5), (10, 15)]
        addRange(3, 12) -> merge to [(1, 15)]
        """
        # Find intervals that overlap with [left, right)
        start = self.ranges.bisect_left((left, left))
        end = self.ranges.bisect_right((right, right))
        
        # Merge overlapping intervals
        merge_left = left
        merge_right = right
        
        # Check if previous interval overlaps
        if start > 0 and self.ranges[start - 1][1] >= left:
            start -= 1
        
        # Expand merge boundaries
        if start < end:
            merge_left = min(merge_left, self.ranges[start][0])
            merge_right = max(merge_right, self.ranges[end - 1][1])
        
        # Remove old intervals
        for _ in range(end - start):
            self.ranges.pop(start)
        
        # Add merged interval
        self.ranges.add((merge_left, merge_right))

    def queryRange(self, left: int, right: int) -> bool:
        """
        Time: O(log n)
        
        Check if [left, right) is fully covered
        """
        idx = self.ranges.bisect_right((left, float('inf')))
        if idx == 0:
            return False
        
        # Check if previous interval covers [left, right)
        interval_left, interval_right = self.ranges[idx - 1]
        return interval_left <= left and right <= interval_right

    def removeRange(self, left: int, right: int) -> None:
        """
        Time: O(n) worst case
        
        Example:
        ranges = [(1, 15)]
        removeRange(3, 10) -> split to [(1, 3), (10, 15)]
        """
        start = self.ranges.bisect_left((left, left))
        end = self.ranges.bisect_right((right, right))
        
        # Check if previous interval overlaps
        if start > 0 and self.ranges[start - 1][1] > left:
            start -= 1
        
        new_intervals = []
        
        for i in range(start, end):
            interval_left, interval_right = self.ranges[i]
            
            # Add left remainder
            if interval_left < left:
                new_intervals.append((interval_left, left))
            
            # Add right remainder
            if right < interval_right:
                new_intervals.append((right, interval_right))
        
        # Remove old intervals
        for _ in range(end - start):
            self.ranges.pop(start)
        
        # Add new split intervals
        for interval in new_intervals:
            self.ranges.add(interval)

---

# Problem 5: Count of Range Sum
**Difficulty**: Hard  
**Link**: https://leetcode.com/problems/count-of-range-sum/

## Problem
Given an array, count the number of range sums that lie in [lower, upper].

## Approach
Use prefix sums + SortedList. For each position, find how many previous prefix sums create a valid range.

## Pattern Recognition
- Prefix sum technique
- Range counting queries
- Use bisect_left and bisect_right for counting

In [ ]:
# https://leetcode.com/problems/count-of-range-sum/
from sortedcontainers import SortedList
from typing import List

class Solution:
    def countRangeSum(self, nums: List[int], lower: int, upper: int) -> int:
        """
        Time: O(n log n)
        Space: O(n)
        
        Key Insight:
        For range sum nums[i:j+1] to be in [lower, upper]:
        lower <= prefix[j] - prefix[i-1] <= upper
        
        Rearranging:
        prefix[j] - upper <= prefix[i-1] <= prefix[j] - lower
        
        So for each j, count how many previous prefix sums are in this range.
        
        Example:
        nums = [-2, 5, -1], lower = -2, upper = 2
        prefix = [0, -2, 3, 2]
        
        j=0: prefix[0]=0, range=[-2, 2], count prefixes in [-2, 2] -> 1 (prefix[0]=0)
        j=1: prefix[1]=-2, range=[-4, 0], count prefixes in [-4, 0] -> 1 (prefix[0]=0)
        j=2: prefix[2]=3, range=[1, 5], count prefixes in [1, 5] -> 1 (prefix[2]=3)
        j=3: prefix[3]=2, range=[0, 4], count prefixes in [0, 4] -> 2 (prefix[0]=0, prefix[2]=3)
        """
        sorted_prefix = SortedList([0])  # Start with prefix[0] = 0
        prefix_sum = 0
        count = 0
        
        for num in nums:
            prefix_sum += num
            
            # Find range of valid previous prefix sums
            # We need: prefix_sum - upper <= prev_prefix <= prefix_sum - lower
            left = sorted_prefix.bisect_left(prefix_sum - upper)
            right = sorted_prefix.bisect_right(prefix_sum - lower)
            
            # Count of valid ranges ending at current position
            count += right - left
            
            # Add current prefix sum for future iterations
            sorted_prefix.add(prefix_sum)
        
        return count

In [ ]:
# https://leetcode.com/problems/time-based-key-value-store/
from sortedcontainers import SortedDict

class TimeMap:
        def __init__(self):
            """
            Initialize your data structure here.
            """
            self.store = {}
    
        def set(self, key: str, value: str, timestamp: int) -> None:
            if key not in self.store:
                self.store[key] = SortedDict()
            
            self.store[key][timestamp] = value
    
        def get(self, key: str, timestamp: int) -> str:
            if key not in self.store:
                return ''
            """
                This method returns the first to the right
            """
            #it = self.store[key].bisect_right(timestamp)
            it = self.store[key].bisect_left(timestamp)

            print(it)

            return self.store[key].peekitem(it - 1)[1] if it else ''

kv = TimeMap()
kv.set("foo", "bar", 1) # store the key "foo" and value "bar" along with timestamp = 1
kv.set("foo", "bar mine", 2) # this is the most closer value
kv.set("foo", "bar2", 6)
kv.set("foo", "bar3", 18)
kv.get("foo", 5) 

In [ ]:
# https://leetcode.com/problems/lru-cache/
import collections

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.hash = collections.OrderedDict()

    def get(self, key: int) -> int:
        if key not in self.hash:
            return -1
        val = self.hash.get(key)
        self.hash.move_to_end(key)
        return val

    def put(self, key: int, value: int) -> None:
        self.hash[key] = value
        self.hash.move_to_end(key)
        if self.capacity < len(self.hash):
            self.hash.popitem(False)

    def get_max_key(self) -> int:
        if not self.hash:
            return -1  # or raise an exception if preferred
        return max(self.hash, key=lambda k: self.hash[k])

# Example usage
cache = LRUCache(2)
cache.put(1, 10)
cache.put(2, 20)
print(cache.get_max_key())  # Output: 2
cache.put(3, 15)
print(cache.get_max_key())  # Output: 3e)


# Your LRUCache object will be instantiated and called as such:
# obj = LRUCache(capacity)
# param_1 = obj.get(key)

---

# Additional Practice Problems

## Beginner
1. **[220. Contains Duplicate III](https://leetcode.com/problems/contains-duplicate-iii/)** - Sliding window + SortedList
2. **[729. My Calendar I](https://leetcode.com/problems/my-calendar-i/)** - Basic interval scheduling
3. **[731. My Calendar II](https://leetcode.com/problems/my-calendar-ii/)** - Double booking allowed

## Intermediate
4. **[295. Find Median from Data Stream](https://leetcode.com/problems/find-median-from-data-stream/)** - Dynamic median
5. **[352. Data Stream as Disjoint Intervals](https://leetcode.com/problems/data-stream-as-disjoint-intervals/)** - Interval merging
6. **[855. Exam Room](https://leetcode.com/problems/exam-room/)** - Maximize distance

## Advanced
7. **[715. Range Module](https://leetcode.com/problems/range-module/)** - Complex interval operations
8. **[327. Count of Range Sum](https://leetcode.com/problems/count-of-range-sum/)** - Prefix sum + bisect
9. **[1157. Online Majority Element In Subarray](https://leetcode.com/problems/online-majority-element-in-subarray/)** - Binary indexed tree + SortedList
10. **[732. My Calendar III](https://leetcode.com/problems/my-calendar-iii/)** - K-booking (like meeting rooms)

---

## Key Takeaways

1. **Use SortedList when**:
   - You need to maintain sorted order with frequent insertions/deletions
   - You need O(log n) search/insert/delete instead of O(n log n) repeated sorts
   - You need to find floor/ceiling/range of values efficiently

2. **Common Patterns**:
   - Sliding window + sorted order (Contains Duplicate III)
   - Interval scheduling/merging (My Calendar, Range Module)
   - Dynamic median/kth element (Median Finder)
   - Prefix sum + bisect (Count of Range Sum)

3. **bisect_left vs bisect_right**:
   - `bisect_left`: Insert **before** existing values (for >= queries)
   - `bisect_right`: Insert **after** existing values (for > queries)
   - Use both to count elements in range: `count = bisect_right(high) - bisect_left(low)`

## Time Complexity Comparison

| Operation | Regular List | SortedList |
|-----------|--------------|------------|
| Insert | O(n) | O(log n) |
| Remove | O(n) | O(log n) |
| Search | O(n) or O(log n) if sorted | O(log n) |
| Access by index | O(1) | O(log n) |
| Min/Max | O(n) or O(1) if sorted | O(1) |
| Sort | O(n log n) | Always sorted |

---